In [20]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2
from collections import deque
from qiskit.quantum_info import Statevector, state_fidelity

# Backend
backend = FakeGuadalupeV2()

# Original circuit
qc_original = QuantumCircuit.from_qasm_file("/Users/nirvanfernando/Desktop/QHD:QC/qhd/lab3/qasm_files/phaseest.qasm")

# Check hardware capacity
if qc_original.num_qubits > backend.num_qubits:
    raise ValueError("The circuit requires more qubits than the backend provides.")

print(f"Logical qubits: {qc_original.num_qubits}")
print(f"Physical qubits: {backend.num_qubits}")
print(f"Original gates: {qc_original.count_ops()}")
print("Native operations:", backend.operation_names)

Logical qubits: 5
Physical qubits: 16
Original gates: OrderedDict([('rz', 42), ('cx', 42), ('h', 8)])
Native operations: ['reset', 'delay', 'x', 'id', 'rz', 'measure', 'sx', 'cx']


In [21]:
def emit_h(qc, q):
    """Decompose H into native RZ and SX gates (up to global phase)."""
    qc.rz(np.pi / 2, q)
    qc.sx(q)
    qc.rz(np.pi / 2, q)


def emit_rx(qc, theta, q):
    """Identity: Rx(theta) = H Rz(theta) H."""
    emit_h(qc, q)
    qc.rz(theta, q)
    emit_h(qc, q)


def emit_ry(qc, theta, q):
    """Decompose Ry(theta) using Rx and Rz."""
    qc.rz(-np.pi / 2, q)
    emit_rx(qc, theta, q)
    qc.rz(np.pi / 2, q)


def emit_u3(qc, theta, phi, lam, q):
    """Decompose U3(theta, phi, lambda) up to global phase."""
    qc.rz(lam, q)
    emit_ry(qc, theta, q)
    qc.rz(phi, q)


def emit_ccx(qc, a, b, c):
    """Decompose Toffoli into native RZ, SX and CX gates."""

    emit_h(qc, c)

    qc.cx(b, c)
    qc.rz(-np.pi / 4, c)

    qc.cx(a, c)
    qc.rz(np.pi / 4, c)

    qc.cx(b, c)
    qc.rz(-np.pi / 4, c)

    qc.cx(a, c)
    qc.rz(np.pi / 4, b)
    qc.rz(np.pi / 4, c)

    emit_h(qc, c)

    qc.cx(a, b)
    qc.rz(np.pi / 4, a)
    qc.rz(-np.pi / 4, b)
    qc.cx(a, b)

def emit_swap(qc, p1, p2):
    qc.cx(p1, p2)
    qc.cx(p2, p1)
    qc.cx(p1, p2)
    


In [22]:

def translate_to_native(qc_original: QuantumCircuit, backend):

    qc_native = qc_original.copy_empty_like(name="Compiler_circuit")

    phase_gates = {
        "z": np.pi,
        "s": np.pi / 2,
        "sdg": -np.pi / 2,
        "t": np.pi / 4,
        "tdg": -np.pi / 4
    }

    for instruction in qc_original.data:

        op = instruction.operation
        name = op.name
        params = op.params

        # 1. Native gates and barriers
        if name in backend.operation_names or name == "barrier":
            qc_native.append(instruction)

        # 2. Hadamard
        elif name == "h":
            q = instruction.qubits[0]
            emit_h(qc_native, q)

        # 3. Phase gates
        elif name in phase_gates:
            q = instruction.qubits[0]
            angle = phase_gates[name]
            qc_native.rz(angle, q)

        # 4. Rx
        elif name == "rx":
            q = instruction.qubits[0]
            theta = params[0]
            emit_rx(qc_native, theta, q)

        # 5. Ry
        elif name == "ry":
            q = instruction.qubits[0]
            theta = params[0]
            emit_ry(qc_native, theta, q)

        # 6. U3 / U
        elif name in ("u", "u3"):
            q = instruction.qubits[0]
            theta, phi, lam = params
            emit_u3(qc_native, theta, phi, lam, q)

        # 7. Toffoli
        elif name == "ccx":
            a, b, c = instruction.qubits
            emit_ccx(qc_native, a, b, c)

        # 8. Unsupported operations
        else:
            raise NotImplementedError(
                f"Gate {name} non ancora supportata"
            )

    return qc_native


In [23]:
qc_native = translate_to_native(qc_original, backend)

print("Original:", qc_original.count_ops())
print("Native:", qc_native.count_ops())


Original: OrderedDict([('rz', 42), ('cx', 42), ('h', 8)])
Native: OrderedDict([('rz', 58), ('cx', 42), ('sx', 8)])


In [24]:
def build_neighbors(backend):

    neighbors = {
        i: set() for i in range(backend.num_qubits)
    }

    for a , b in backend.coupling_map.get_edges():
        neighbors[a].add(b)
        neighbors[b].add(a)

    return neighbors


def shortest_path(start, goal, neighbors):
    """Find the shortest path between two physical qubits using BFS."""

    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        current, path = queue.popleft()

        if current == goal:
            return path

        for next in sorted(neighbors[current]):
            if next not in visited:
                visited.add(next)
                queue.append((next, path + [next]))

    raise ValueError("No path between qubits {start} and {goal}")


In [25]:
def initialize_layout(qc_native, backend):
    """Assign logical qubits to trivial porision in physical qubit."""

    if qc_native.num_qubits > backend.num_qubits:
        raise ValueError("Not enough physical qubits.")

    return {
        p: qc_native.qubits[p] if p < qc_native.num_qubits else None
        for p in range(backend.num_qubits)
    }


def find_physical(logical_qubit, layout):
    """Find the physical position of a logical qubit."""

    for physical, logical in layout.items():
        if logical == logical_qubit:
            return physical

    raise ValueError("Logical qubit not found")


def update_layout(layout, p1, p2):
    """Update the layout after swapping two physical positions."""

    layout[p1], layout[p2] = layout[p2], layout[p1]

In [26]:
def route_circuit(qc_native: QuantumCircuit, backend):

    layout = initialize_layout(qc_native, backend)

    neighbors = build_neighbors(backend)
    edges = set(backend.coupling_map.get_edges())

    qc_physical = QuantumCircuit(backend.num_qubits)

    for instruction in qc_native.data:
        name = instruction.operation.name

        physical_qubits = [
            find_physical(q, layout)
            for q in instruction.qubits
        ]

        if name == "cx":
            control, target = physical_qubits

            # Routing: move the control towards the target 
            if (control, target) not in edges:
                path = shortest_path(control, target, neighbors)

                for i in range(len(path) - 2):
                    p1 = path[i]
                    p2 = path[i +1]

                    emit_swap(qc_physical, p1, p2)
                    update_layout(layout, p1, p2)

                control = path[-2]
                target = path[-1]

            qc_physical.cx(control, target)

        else:
            qc_physical.append(instruction.operation, physical_qubits)

    return qc_physical, layout

In [27]:
qc_physical, final_layout = route_circuit(qc_native, backend)

print("Physical circuit:", qc_physical.count_ops())

native_gates = set(backend.operation_names)

non_native = set(qc_physical.count_ops()) - native_gates

print("Non-native gates:", non_native)

Physical circuit: OrderedDict([('cx', 84), ('rz', 58), ('sx', 8)])
Non-native gates: set()


In [29]:
# Find the final physical position of each logical qubit
final_positions = [
    find_physical(q, final_layout)
    for q in qc_native.qubits
]

print("Final positions:", final_positions)

# Original circuit represented on the physical register
qc_reference = QuantumCircuit(backend.num_qubits)

qc_reference.compose(
    qc_original,
    qubits=final_positions,
    inplace=True
)

# Compute statevectors
psi_reference = Statevector.from_instruction(qc_reference)
psi_physical = Statevector.from_instruction(qc_physical)

# Fidelity
fidelity = state_fidelity(psi_reference, psi_physical)

print(f"Fidelity: {fidelity}")

Final positions: [0, 1, 2, 3, 4]
Fidelity: 0.9999999999999956
